# 01 · Limpieza y delimitación de la población

**Objetivo de la tesis:** predecir el **deterioro temprano de pago** en **clientes independientes sin historial de pago con la entidad** (primer crédito) que compran **motocicleta** a crédito, con la información disponible al momento de la solicitud. El evento principal es la **mora de 30+ días en los primeros 6 meses**; la mora 30+ en 3 meses se usa como monitoreo, y la de 60+ y 90+ como confirmación.

Este notebook documenta, con las salidas reales de los datos:

1. La estructura de la base y sus controles de integridad.
2. Tres hallazgos que condicionan toda la limpieza: la sobrescritura de atributos, el corte reextraído y el cambio de significado de la zona.
3. El **embudo de población**, paso a paso y con la justificación de cada filtro.
4. La **variable objetivo**, sus ventanas y el tratamiento de los créditos que salen del panel sin deterioro, que son pagos anticipados y cuentan como buenos pagadores.
5. La comparación con la tabla de población de referencia.

La lógica está en `src/limpieza.py`. Aquí se replica paso a paso y se verifica que el resultado sea idéntico al del script.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
import limpieza as L
import protocolo as P

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.3f}".format)

df = pd.read_parquet(RAIZ / "datos" / "cartera.parquet")
fecha = L.fecha_excel

## 1. Estructura de la base

Cada fila es **una obligación (crédito) en un mes de corte**. Un cliente puede tener varias obligaciones, y cada obligación aparece una vez por mes mientras esté vigente. Las fechas vienen como número serial de Excel.

In [2]:
print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}")
print(f"Obligaciones: {df['Numero_Encriptado'].nunique():,}   Clientes: {df['Cedula_Encriptado'].nunique():,}")
print(f"Cortes: {fecha(df['Corte']).min():%Y-%m} a {fecha(df['Corte']).max():%Y-%m} ({df['Corte'].nunique()} meses)")

Filas: 360,769   Columnas: 77
Obligaciones: 38,070   Clientes: 32,082


Cortes: 2022-11 a 2026-01 (39 meses)


In [3]:
# Filas por corte: la cartera crece mes a mes
por_corte = df.groupby(fecha(df["Corte"]).dt.strftime("%Y-%m")).agg(filas=("Numero_Encriptado", "size"))
por_corte.T

Corte,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01
filas,28,93,132,221,399,475,780,1207,1788,2307,3211,3862,4380,4878,5470,6139,6777,7513,8067,8494,8961,9306,9974,10646,11356,12173,12745,13621,14378,15247,16063,16943,18079,18867,19904,20594,21231,22046,22414


## 2. Controles de integridad

In [4]:
controles = {
    "Filas exactamente duplicadas": df.duplicated().sum(),
    "Obligación repetida en un mismo corte": df.duplicated(["Numero_Encriptado", "Corte"]).sum(),
    "Flag_Over_30 distinto de (Dias Mora > 30)": (df["Flag_Over_30"] != (df["Dias Mora"] > 30)).sum(),
    "Flag_Over_60 distinto de (Dias Mora > 60)": (df["Flag_Over_60"] != (df["Dias Mora"] > 60)).sum(),
    "Flag_Over_90 distinto de (Dias Mora > 90)": (df["Flag_Over_90"] != (df["Dias Mora"] > 90)).sum(),
    "Obligaciones cuyo primer registro no es C-0": (df.groupby("Numero_Encriptado")["Mes_Cosecha"].min() > 0).sum(),
    "Marca_Cosecha distinto de Flag_Over_30 (columna duplicada)": (df["Marca_Cosecha"] != df["Flag_Over_30"]).sum(),
}
pd.Series(controles, name="casos").to_frame()

,casos
Filas exactamente duplicadas,0
Obligación repetida en un mismo corte,0
Flag_Over_30 distinto de (Dias Mora > 30),0
Flag_Over_60 distinto de (Dias Mora > 60),0
Flag_Over_90 distinto de (Dias Mora > 90),0
Obligaciones cuyo primer registro no es C-0,0
Marca_Cosecha distinto de Flag_Over_30 (columna duplicada),0


In [5]:
# Columnas vacías o constantes
vacias = df.isna().mean().loc[lambda s: s > 0.95].rename("% nulos")
constantes = df.nunique().loc[lambda s: s <= 1].rename("valores distintos")
display(vacias.to_frame(), constantes.to_frame())

,% nulos
PROFESSION,0.975
Ultimo Pago,1.000
Mentorías,1.000


,valores distintos
Estado Fallecido,1
Otros Cobros,1
Ultimo Pago,0
Mentorías,0


**Lectura:** la base es íntegra (sin duplicados y con indicadores de mora coherentes). `Ultimo Pago`, `Mentorías` y `PROFESSION` están vacías o casi vacías, `Otros Cobros` y `Estado Fallecido` son constantes, y `Marca_Cosecha` es una copia de `Flag_Over_30`. Ninguna de ellas se usa.

## 3. Los atributos de la solicitud se sobrescriben

Variables como el tipo de moto, el precio, el estrato o el sector económico deberían ser fijas desde el desembolso. Se mide en qué porcentaje de las obligaciones cambian de un corte a otro (créditos de motos a personas).

In [6]:
motos = df[(df["Tipo"] == "01CR") & (df["Productos"] == "Motocicletas")].sort_values(["Numero_Encriptado", "Corte"])
atributos = ["TIPO_MOTO", "Product_Price", "Estrato", "SECTOR_ECONOMICO", "Nivel Estudio", "Ciudad",
             "Nombre de la tienda", "Zona", "Sexo", "Estado Civil", "Segment"]
cambia = {a: (motos.groupby("Numero_Encriptado")[a].nunique(dropna=False) > 1).mean() for a in atributos}
pd.Series(cambia, name="% obligaciones donde cambia").sort_values(ascending=False).to_frame()

,% obligaciones donde cambia
Nombre de la tienda,0.695
Zona,0.636
TIPO_MOTO,0.631
Product_Price,0.608
Estrato,0.305
Nivel Estudio,0.285
Ciudad,0.214
SECTOR_ECONOMICO,0.141
Segment,0.005
Estado Civil,0.000


In [7]:
# ¿En qué corte ocurren los cambios de TIPO_MOTO?
m = motos[["Numero_Encriptado", "Corte", "TIPO_MOTO"]].copy()
m["anterior"] = m.groupby("Numero_Encriptado")["TIPO_MOTO"].shift()
cambios = m[m["anterior"].notna() & (m["anterior"] != m["TIPO_MOTO"])]
fecha(cambios["Corte"]).dt.strftime("%Y-%m").value_counts().head(6).rename("cambios de moto").to_frame()

,cambios de moto
Corte,
2026-01,14637
2025-04,12669
2025-07,1642
2025-09,112
2023-07,88
2023-11,87


In [8]:
# Ejemplo: una obligación que 'cambia' de moto y de precio
ejemplo = cambios["Numero_Encriptado"].iloc[0]
e = motos[motos["Numero_Encriptado"] == ejemplo][["Corte", "Mes_Cosecha", "TIPO_MOTO", "Product_Price", "Estrato", "Valor Capital"]]
# Se muestra solo el primer corte de cada "versión" distinta de los atributos
e = e.assign(Corte=fecha(e["Corte"]).dt.strftime("%Y-%m"))
e = e.drop_duplicates(["TIPO_MOTO", "Product_Price", "Estrato"])
# Por confidencialidad, cada escritura del modelo se muestra como «escritura 1», «escritura 2»…
e.assign(TIPO_MOTO=e["TIPO_MOTO"].map({v: f"escritura {k + 1}" for k, v in enumerate(e["TIPO_MOTO"].unique())}))

,Corte,Mes_Cosecha,TIPO_MOTO,Product_Price,Estrato,Valor Capital
17689,2023-09,0,escritura 1,"8,400,000.000",3,"7,922,508.000"
182817,2025-04,19,escritura 2,"8,400,000.000",3,"7,922,508.000"
358045,2026-01,28,escritura 1,"16,100,000.000",3,"7,922,508.000"


**Decisión:** todas las variables explicativas se toman del **registro C-0** (mes de desembolso). Es la única foto de lo que se sabía al aprobar el crédito. En la fila de ejemplo el capital prestado no cambia, pero la moto y su precio sí: el sistema cruzó al cliente con los datos de otra solicitud.

## 4. El corte de enero de 2026 está reextraído

Ese corte concentra la mayor cantidad de atributos reescritos. Se revisa también si su **mora** es coherente con la de diciembre, para saber si el problema afecta al desempeño o solo a los atributos.

In [9]:
ultimo = df["Corte"].max()
p = df[df["Corte"] >= ultimo - 31].pivot_table(index="Numero_Encriptado", columns="Corte", values="Dias Mora").dropna()
dic, ene = p.iloc[:, 0], p.iloc[:, 1]
print(f"Créditos presentes en dic-2025 y ene-2026: {len(p):,}")
print(f"Mora que sube más de 31 días en un mes (imposible): {((ene - dic) > 31).sum()}")
solo_ultimo = df.groupby("Cedula_Encriptado")["Corte"].min().eq(ultimo).sum()
print(f"Clientes que solo aparecen en ene-2026: {solo_ultimo:,}")

Créditos presentes en dic-2025 y ene-2026: 21,096
Mora que sube más de 31 días en un mes (imposible): 4
Clientes que solo aparecen en ene-2026: 1,199


**Decisión:** se descarta el corte completo. Su mora es coherente (solo unos pocos saltos imposibles), pero sus atributos no son confiables. Por consistencia no se mezclan fuentes. Los clientes que salen son desembolsos de enero de 2026, que igual no tendrían 6 meses de historia. **El último corte válido pasa a ser diciembre de 2025.**

## 5. El código de zona cambió de significado

Normalizar el formato (`'05` = `5`) no basta: el mismo código corresponde a regiones distintas en distintos años. El nombre de cada zona incluye también el de la persona a cargo; por protección de datos personales (Ley 1581 de 2012), la tabla muestra solo las regiones y la zona.

In [10]:
import unicodedata

REGIONES = {"AMAZONAS", "ANTIOQUIA", "ARAUCA", "ATLANTICO", "BOLIVAR", "BOYACA", "CALDAS", "CAQUETA", "CASANARE", "CAUCA",
            "CESAR", "CHOCO", "CORDOBA", "CUNDINAMARCA", "GUAINIA", "GUAVIARE", "HUILA", "GUAJIRA", "MAGDALENA", "META",
            "NARINO", "NORTE", "SANTANDER", "SANTANDERES", "PUTUMAYO", "QUINDIO", "RISARALDA", "SUCRE", "TOLIMA", "VALLE",
            "DEL", "VAUPES", "VICHADA", "BOGOTA", "EJE", "CAFETERO", "URABA", "MONTERIA", "BAJO", "LLANOS", "COSTA", "SUR",
            "CENTRO", "ORIENTE", "OCCIDENTE", "ZONA", "LA", "DE"}


def zona_sin_nombres(valor):
    """«Nombre Zona» sin los nombres de las personas a cargo: en 2024 el nombre va antes de las regiones
    («nombre- regiones- ZONA …») y en 2025-2026 va al final («regiones- nombre»). Quedan solo las regiones y la zona."""
    if not isinstance(valor, str) or "- " not in valor:
        return valor
    seg = [s.strip() for s in valor.split("- ")]
    cola = seg.pop() if not seg[-1].startswith("ZONA") else None  # 2025-2026: el nombre va al final
    if len(seg) >= 3:  # 2024: el nombre va al principio
        seg = seg[1:]
    plano = lambda s: unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().upper()
    seg = [s for s in seg if "/" in s or s.startswith("ZONA") or all(plano(t) in REGIONES for t in s.split())]
    if not seg and cola:  # queda solo la parte de región de la cola (p. ej., la ciudad)
        seg = [" ".join(t for t in cola.split() if plano(t) in REGIONES)]
    return "- ".join(s for s in seg if s) or "(sin región)"


c0_motos = motos[motos["Mes_Cosecha"] == 0].copy()
c0_motos["zona"] = L.codigo_entero(c0_motos["Zona"])
c0_motos["anio"] = fecha(c0_motos["Corte"]).dt.year
c0_motos["ciudad"] = L.normalizar_texto(c0_motos["Ciudad"])
# «Nombre Zona» incluye el nombre de la persona a cargo de la zona: se muestran solo las regiones y la zona
nombres = (c0_motos.groupby(["zona", "anio"])["Nombre Zona"].agg(lambda s: s.mode().iloc[0]).unstack()
           .map(zona_sin_nombres))
nombres

anio,2023,2024,2025,2026
zona,,,,
1,CAQUETA/ PUTUMAYO- ZONA SUR,CAQUETA/ PUTUMAYO- ZONA SUR,SIN ASIGNAR,NaN
2,SANTANDERES/ CESAR- ZONA ORIENTE,SANTANDERES/ CESAR- ZONA ORIENTE,SANTANDERES/ CESAR/ BOLIVAR/ ATLANTICO,SANTANDERES/ CESAR/ BOLIVAR/ ATLANTICO
3,VALLE DEL CAUCA/ HUILA- ZONA SUR OCCIDENTE,VALLE DEL CAUCA/ HUILA- ZONA SUR OCCIDENTE,VALLE/ CAUCA/ HUILA/ NARINO,VALLE/ CAUCA/ HUILA/ NARINO
4,EJE CAFETERO/ TOLIMA- ZONA CENTRO OCCIDENTE,EJE CAFETERO/ TOLIMA- ZONA CENTRO OCCIDENTE,EJE CAFETERO/ TOLIMA,EJE CAFETERO/ TOLIMA
5,ANTIOQUIA- ZONA OCCIDENTE,ANTIOQUIA- ZONA OCCIDENTE,ANTIOQUIA/ CAQUETA,ANTIOQUIA/ CAQUETA
6,CUNDINAMARCA/ BOYACA- ZONA CENTRO,CUNDINAMARCA/ BOYACA- ZONA CENTRO,CUNDINAMARCA/ BOYACA/ META,CUNDINAMARCA/ BOYACA/ META
7,META/ CASANARE/ ATLANTICO- ZONA SUR ORIENTE,META/ CASANARE/ ATLANTICO,SIN ASIGNAR 2,NaN
8,NaN,MONTERIA,MONTERIA/ URABA/ BAJO CAUCA/ PUTUMAYO,MONTERIA/ URABA/ BAJO CAUCA/ PUTUMAYO
11,NaN,NaN,NaN,MUSICAL


In [11]:
# La misma ciudad cambia de zona según el año
pd.crosstab([c0_motos["ciudad"], c0_motos["anio"]], c0_motos["zona"]).loc[["MEDELLIN", "BOGOTA", "CALI"]]

zona             1   2     3   4     5     6   7   8   11
ciudad   anio                                            
MEDELLIN 2023  1340   0     0   0    22     1   0   0   0
         2024  1203   0     4   2   774     1   0   0   0
         2025   117   4     0  12  2624     0   0  13   0
         2026     0  21    24  17    86    28   0   7   3
BOGOTA   2023     1   0     1   0     2  1149   0   0   0
         2024     1   1     7   5     6  1704   3   0   0
         2025     0   6     9   6     2  1639   2   6   0
         2026     0  13    21  17    28    43   0   8   7
CALI     2023     0   0   452  72     0     0   0   0   0
         2024     2   0  1004  43     1     1   0   0   0
         2025     2   0  1242   2     0     3   0   0   0
         2026     0  11    24   8    19    15   0   3   1

**Decisión:** no se usa la `Zona` original. Si se usara, el modelo aprendería "zona 1 = mora alta" solo porque la zona 1 existió cuando la mora era alta (2023). Se crea `zona_homologada`: a cada ciudad se le asigna la zona que tuvo en 2025, que es la estructura vigente. Se detalla en el notebook 02.

## 6. Embudo de población, paso a paso

Cada celda aplica un filtro y registra cuántas filas, obligaciones y clientes quedan.

In [12]:
embudo = []
def registrar(paso, d, motivo):
    embudo.append({"paso": paso, "filas": len(d), "obligaciones": d["Numero_Encriptado"].nunique(),
                   "clientes": d["Cedula_Encriptado"].nunique(), "motivo": motivo})
    return pd.DataFrame(embudo).set_index("paso")[["filas", "obligaciones", "clientes"]].tail(2)

p0 = df
registrar("1. Base cruda", p0, "una fila por obligación y mes de corte")

,filas,obligaciones,clientes
paso,,,
1. Base cruda,360769,38070,32082


### Paso 2 · Sin el corte reextraído (enero 2026)

In [13]:
p1 = p0[p0["Corte"] < p0["Corte"].max()]
ultimo_corte = p1["Corte"].max()
print("Último corte válido:", fecha(pd.Series([ultimo_corte]))[0].strftime("%Y-%m"))
registrar("2. Sin el corte reextraído", p1, "reescribe atributos del cliente y del contrato")

Último corte válido: 2025-12


,filas,obligaciones,clientes
paso,,,
1. Base cruda,360769,38070,32082
2. Sin el corte reextraído,338355,36752,30883


### Paso 3 · Primer crédito de cada cliente

La población **no tiene historial de pago con la entidad**: desde su segundo crédito, la entidad ya sabe cómo paga el cliente. El primer crédito se busca en **toda la cartera**, antes de filtrar producto o sector. Un cliente cuyo primer crédito fue otro producto ya tiene historial al comprar la moto.

**Créditos del mismo día:** si un cliente tiene varios créditos desembolsados el mismo día, se toma el de menor identificador. Es arbitrario pero reproducible, y no usa variables del contrato.

**Alcance:** el panel empieza en noviembre de 2022 y todos los créditos aparecen desde su mes de desembolso. «Primer crédito» significa, por tanto, el primero observado en el panel; la base no permite saber si un cliente tuvo créditos anteriores a esa fecha.

In [14]:
creditos = p1[["Cedula_Encriptado", "Numero_Encriptado", "Fecha desembolso"]].drop_duplicates()
primer_dia = creditos.groupby("Cedula_Encriptado")["Fecha desembolso"].transform("min")
mismo_dia = creditos[creditos["Fecha desembolso"] == primer_dia].groupby("Cedula_Encriptado").size()
print(f"Clientes con más de un crédito: {(creditos.groupby('Cedula_Encriptado').size() > 1).sum():,}")
print(f"Clientes con 2+ créditos el mismo día de su primer desembolso: {(mismo_dia > 1).sum()}")

primeros = (creditos.sort_values(["Cedula_Encriptado", "Fecha desembolso", "Numero_Encriptado"])
            .drop_duplicates("Cedula_Encriptado")["Numero_Encriptado"])
p2 = p1[p1["Numero_Encriptado"].isin(primeros)]
registrar("3. Primer crédito de cada cliente", p2, "los créditos siguientes ya tienen historial de pago con la entidad")

Clientes con más de un crédito: 297
Clientes con 2+ créditos el mismo día de su primer desembolso: 30


,filas,obligaciones,clientes
paso,,,
2. Sin el corte reextraído,338355,36752,30883
3. Primer crédito de cada cliente,327011,30883,30883


In [15]:
# Qué sale en este paso: factoring de distribuidores frente a créditos de consumo posteriores al primero
obl = p1.drop_duplicates("Numero_Encriptado")
salen = obl[~obl["Numero_Encriptado"].isin(primeros)]
fw = obl[obl["Tipo"] == "01CW"]
id_fw = fw.groupby("Cedula_Encriptado")["Tipo Identificacion"].agg(lambda s: s.mode().iloc[0])
sal_fw, sal_cr = salen[salen["Tipo"] == "01CW"], salen[salen["Tipo"] == "01CR"]
por_cliente = obl[obl["Tipo"] == "01CR"].groupby("Cedula_Encriptado").size()
print(f"Salen en el paso 3: {len(salen):,} obligaciones")
print(f"   factoring: {len(sal_fw):,}, de {sal_fw['Cedula_Encriptado'].nunique()} identificadores "
      f"(tipo de identificación: {id_fw[id_fw.index.isin(sal_fw['Cedula_Encriptado'])].value_counts().to_dict()}); "
      f"hasta {fw.groupby('Cedula_Encriptado').size().max()} obligaciones por identificador")
print(f"   crédito de consumo: {len(sal_cr)} créditos posteriores al primero, de {sal_cr['Cedula_Encriptado'].nunique()} "
      f"clientes ({sal_cr['Productos'].value_counts().to_dict()})")
print("Créditos de consumo por cliente:", por_cliente.value_counts().sort_index().to_dict())
print("Identificadores vacíos:", int(p1["Cedula_Encriptado"].isna().sum()
                                     + (p1["Cedula_Encriptado"].astype(str).str.strip() == "").sum()))
del obl, salen, fw, id_fw, sal_fw, sal_cr, por_cliente

Salen en el paso 3: 5,869 obligaciones
   factoring: 5,627, de 62 identificadores (tipo de identificación: {'NIT': 52, 'CC': 10}); hasta 701 obligaciones por identificador
   crédito de consumo: 242 créditos posteriores al primero, de 235 clientes ({'Motocicletas': 240, 'Instrumentos Musicales': 2})
Créditos de consumo por cliente: {1: 30585, 2: 228, 3: 7}
Identificadores vacíos: 0


**Qué sale en el paso 3.** La diferencia entre obligaciones y clientes no viene de un identificador vacío o genérico: casi todo es factoring, la línea de crédito de los distribuidores, que tiene muchas obligaciones por identificador y se retira en el paso 4. Del crédito de consumo salen solo los créditos posteriores al primero, y ningún cliente tiene más de tres.

### Paso 4 · Sin factoring

In [16]:
fact = p2[p2["Tipo"] == "01CW"].drop_duplicates("Numero_Encriptado")
print("Tipo de identificación en factoring:", fact["Tipo Identificacion"].value_counts().to_dict())
print("Líneas:", fact["Linea de credito"].value_counts().to_dict())
print("Con sector económico informado:", fact["SECTOR_ECONOMICO"].notna().sum())
p3 = p2[p2["Tipo"] == "01CR"]
registrar("4. Sin factoring", p3, "fondeo a distribuidores (NIT), no crédito de consumo")

Tipo de identificación en factoring: {'NIT': 53, 'CC': 10}
Líneas: {'W01MS': 27, 'W01MP': 25, 'W01R': 6, 'W01PE': 5}
Con sector económico informado: 0


,filas,obligaciones,clientes
paso,,,
3. Primer crédito de cada cliente,327011,30883,30883
4. Sin factoring,326911,30821,30821


El factoring (línea `01CW`) financia inventario de distribuidores. No es crédito de consumo a personas y no tiene datos sociodemográficos.

### Paso 5 · Solo motocicletas

In [17]:
c0_p3 = p3[p3["Mes_Cosecha"] == 0].drop_duplicates("Numero_Encriptado")
print(c0_p3.groupby("Productos")["Product_Price"].median().rename("precio mediano").map("{:,.0f}".format).to_string())
p4 = p3[p3["Productos"] == "Motocicletas"]
registrar("5. Solo motocicletas", p4, "instrumentos musicales: otra línea de negocio")

Productos
Instrumentos Musicales     3,650,000
Motocicletas              12,900,000


,filas,obligaciones,clientes
paso,,,
4. Sin factoring,326911,30821,30821
5. Solo motocicletas,324241,30286,30286


### Paso 6 · Independientes al desembolso

Es el filtro central de la tesis. Se usa el sector registrado **en C-0**, porque el sector se sobrescribe después (sección 3). Se conservan independientes formales e informales.

In [18]:
c0_p4 = p4[p4["Mes_Cosecha"] == 0].drop_duplicates("Numero_Encriptado").set_index("Numero_Encriptado")
display(c0_p4["SECTOR_ECONOMICO"].value_counts(dropna=False).rename("créditos en C-0").to_frame())
independientes = c0_p4.index[c0_p4["SECTOR_ECONOMICO"].isin(L.SECTORES_INDEPENDIENTES)]
p5 = p4[p4["Numero_Encriptado"].isin(independientes)]
registrar("6. Independientes al desembolso", p5, "sector económico en el mes de desembolso")

,créditos en C-0
SECTOR_ECONOMICO,
formal_employee,24585
formal_independent,3882
informal_independent,837
pensioner,573
informal_employee,409


,filas,obligaciones,clientes
paso,,,
5. Solo motocicletas,324241,30286,30286
6. Independientes al desembolso,41936,4719,4719


In [19]:
# Verificación: el embudo replicado aquí es idéntico al de src/limpieza.py
panel_script, c0_script, embudo_script, _ = L.filtrar_poblacion(df)
assert set(panel_script["Numero_Encriptado"]) == set(p5["Numero_Encriptado"])
assert [e["clientes"] for e in embudo_script] == [e["clientes"] for e in embudo]
print("OK: el notebook y el script producen la misma población.")

OK: el notebook y el script producen la misma población.


## 7. Variable objetivo

**Definición principal:** el cliente alcanza **30 o más días de mora en algún momento de los meses 1 a 6** después del desembolso (30+ a 6 meses). Es el único objetivo con el que se escogen variables y versiones.

Con el mismo puntaje se reportan, sin usarlas para decidir:
- **30+ a 3 meses** (meses 1 a 3): madura antes y sirve para monitorear. No es independiente de la de 6 meses: para un mismo crédito, sus eventos son un subconjunto de los de 6 meses.
- **60+ y 90+ a 6 meses**: confirmación descriptiva con un deterioro más severo.

Para tener resultado en una ventana de *h* meses, el crédito debe:
1. **Madurez:** haber pasado al menos *h* meses entre el desembolso y el último corte válido (diciembre de 2025).
2. **Observación:** que el panel lo muestre durante esos *h* meses o que salga antes sin haber alcanzado el evento: esos créditos son pagos anticipados y cuentan como buenos pagadores (sección siguiente).

In [20]:
obj = L.construir_objetivos(p5, ultimo_corte)
obj["estado_6m"].value_counts().rename("créditos").to_frame()

,créditos
estado_6m,
observado,2524
inmaduro,1811
sale_al_dia,248
sale_sin_exposicion,103
sale_mora_menor_30,26
sale_con_mora,7


**Nota:** de los 26 créditos que salen con mora menor de 30 días, uno había alcanzado 30+ días antes de salir y cuenta como evento. Así, salen **con deterioro 8** (7 + 1) y **sin deterioro 376** (248 al día, 103 en el mes 0 y 25 con mora menor de 30).

In [21]:
# Ventana de 3 meses y anidamiento con la de 6 meses (los pagos anticipados cuentan como buenos pagadores)
obj = P.agregar_etiquetas(obj)
y6, y3 = obj[f"mora30_6m__{P.ETIQUETA}"], obj[f"mora30_3m__{P.ETIQUETA}"]
print(obj["estado_3m"].value_counts().rename("créditos (ventana de 3 meses)").to_frame())
ambos = obj[y6.notna() & y3.notna()]
ev6 = y6[ambos.index] == 1
print(f"Créditos con resultado a 3 meses: {y3.notna().sum():,} (tasa {y3.mean():.1%})")
print(f"Créditos con resultado a 6 meses: {y6.notna().sum():,} (tasa {y6.mean():.1%})")
print(f"De los eventos 30+ a 6 meses, {y3[ambos.index][ev6].mean():.1%} ya había ocurrido a los 3 meses")
print(f"Eventos a 3 meses que no son eventos a 6 meses: {int(((y3[ambos.index] == 1) & ~ev6).sum())}")

                     créditos (ventana de 3 meses)
estado_3m                                         
observado                                     3430
inmaduro                                       959
sale_sin_exposicion                            170
sale_al_dia                                    145
sale_mora_menor_30                              13
sale_con_mora                                    2
Créditos con resultado a 3 meses: 3,760 (tasa 9.4%)
Créditos con resultado a 6 meses: 2,908 (tasa 18.2%)
De los eventos 30+ a 6 meses, 61.6% ya había ocurrido a los 3 meses
Eventos a 3 meses que no son eventos a 6 meses: 0


### ¿Qué pasa con los créditos que salen del panel antes de los 6 meses?

La base no marca como pagados los créditos que terminan de pagar antes de su fecha: dejan de aparecer en los cortes. Se revisa cómo salen.

In [22]:
ultimos = p5.sort_values("Corte").groupby("Numero_Encriptado").tail(1).set_index("Numero_Encriptado")
salen = obj.index[obj["estado_6m"].str.startswith("sale")]
x = ultimos.loc[salen].join(obj["estado_6m"])
resumen = x.groupby("estado_6m").agg(
    creditos=("Mes_Cosecha", "size"),
    mob_salida_mediano=("Mes_Cosecha", "median"),
    saldo_pendiente_mediano=("Saldo Capital", lambda s: (s / x.loc[s.index, "Valor Capital"]).median()),
    pct_pago_casi_todo=("Saldo Capital", lambda s: (s / x.loc[s.index, "Valor Capital"] < 0.2).mean()),
)
resumen

,creditos,mob_salida_mediano,saldo_pendiente_mediano,pct_pago_casi_todo
estado_6m,,,,
sale_al_dia,248,3.000,0.776,0.173
sale_con_mora,7,3.000,1.000,0.000
sale_mora_menor_30,26,3.000,0.927,0.077
sale_sin_exposicion,103,0.000,1.000,0.019


In [23]:
# ¿Son refinanciaciones? (el cliente recibe otro crédito cerca de la fecha de salida)
al_dia = x[x["estado_6m"] == "sale_al_dia"]
otros = df[df["Cedula_Encriptado"].isin(al_dia["Cedula_Encriptado"]) & ~df["Numero_Encriptado"].isin(al_dia.index)]
nuevo = otros.groupby("Cedula_Encriptado")["Fecha desembolso"].min()
cerca = (al_dia["Cedula_Encriptado"].map(nuevo) - al_dia["Corte"]).abs() <= 62
print(f"Salen al día: {len(al_dia)} | con otro crédito ±2 meses de la salida: {cerca.sum()}")

Salen al día: 248 | con otro crédito ±2 meses de la salida: 0


**Lectura:** los créditos que salen **al día** lo hacen debiendo todavía la mayor parte del capital, que cancelan al salir, y ninguno es una refinanciación (0 de 248 recibieron otro crédito en los dos meses alrededor de la salida). Los créditos que salen sin deterioro son **pagos anticipados**: el cliente terminó de pagar la deuda antes de su fecha.

**Regla del evento:**

| Situación | Resultado | Justificación |
|---|---|---|
| Completa los 6 meses en el panel | Mora observada | Información completa |
| Sale **después de alcanzar** el umbral de mora | **Evento** | El deterioro ya ocurrió dentro de la ventana y se observó |
| Sale **sin haber alcanzado** el umbral (al día, en el mes 0 o con 1–29 días) | **Buen pagador** | Pago anticipado: terminó de pagar antes de su fecha |
| No han pasado 6 meses al último corte | Excluido (inmaduro) | Aún no se puede observar |

En el código, la columna del evento es `mora30_6m__B_todos` (y sus equivalentes a 3 meses, 60+ y 90+), que construye `protocolo.agregar_etiquetas`.

In [24]:
tabla = pd.DataFrame({
    f"{u}+ días": {
        "créditos con resultado": obj[f"mora{u}_6m__{P.ETIQUETA}"].notna().sum(),
        "malos": int(obj[f"mora{u}_6m__{P.ETIQUETA}"].sum()),
        "tasa de mora": obj[f"mora{u}_6m__{P.ETIQUETA}"].mean(),
        "malos que salieron del panel": int(obj.loc[obj["estado_6m"] != "observado", f"mora{u}_6m__{P.ETIQUETA}"].sum()),
        "pagos anticipados (buenos)": int((obj[f"mora{u}_6m"].isna() & obj[f"mora{u}_6m__{P.ETIQUETA}"].notna()).sum()),
    } for u in (30, 60, 90)})
tabla

,30+ días,60+ días,90+ días
créditos con resultado,"2,908.000","2,904.000","2,901.000"
malos,528.000,391.000,292.000
tasa de mora,0.182,0.135,0.101
malos que salieron del panel,8.000,3.000,0.000
pagos anticipados (buenos),376.000,377.000,377.000


In [25]:
# Pagos anticipados (salidas sin deterioro) por año de desembolso (créditos maduros a 6 meses)
anio_des = fecha(p5.drop_duplicates("Numero_Encriptado").set_index("Numero_Encriptado")["Fecha desembolso"]).dt.year
o = obj.join(anio_des.rename("anio"))
mad = o[o["estado_6m"] != "inmaduro"]
sale = mad["estado_6m"].str.startswith("sale") & mad["mora30_6m"].isna()
t = pd.DataFrame({"maduros a 6 meses": mad.groupby("anio").size(), "pagos anticipados": sale.groupby(mad["anio"]).sum(),
                  "de ellas en el mes 0": (sale & (mad["estado_6m"] == "sale_sin_exposicion")).groupby(mad["anio"]).sum()})
t["% de pagos anticipados"] = (t["pagos anticipados"] / t["maduros a 6 meses"]).map("{:.1%}".format)
t

,maduros a 6 meses,pagos anticipados,de ellas en el mes 0,% de pagos anticipados
anio,,,,
2023,528,47,13,8.9%
2024,1347,145,32,10.8%
2025,1033,184,58,17.8%


**Lectura:** los pagos anticipados aumentan en 2025: 17,8 % de los créditos maduros a 6 meses, frente a 8,9 % en 2023 y 10,8 % en 2024.

## 8. Embudo final

In [26]:
madura = obj[obj["estado_6m"] != "inmaduro"]
embudo.append({"paso": "7. Con 6 meses cumplidos", "obligaciones": len(madura), "clientes": len(madura),
               "motivo": "créditos recientes: aún no cumplen 6 meses"})
final = pd.DataFrame(embudo).set_index("paso")
final["salen (clientes)"] = final["clientes"].shift() - final["clientes"]
final

,filas,obligaciones,clientes,motivo,salen (clientes)
paso,,,,,
1. Base cruda,"360,769.000",38070,32082,una fila por obligación y mes de corte,NaN
2. Sin el corte reextraído,"338,355.000",36752,30883,reescribe atributos del cliente y del contrato,"1,199.000"
3. Primer crédito de cada cliente,"327,011.000",30883,30883,los créditos siguientes ya tienen historial de pago con la entidad,0.000
4. Sin factoring,"326,911.000",30821,30821,"fondeo a distribuidores (NIT), no crédito de consumo",62.000
5. Solo motocicletas,"324,241.000",30286,30286,instrumentos musicales: otra línea de negocio,535.000
6. Independientes al desembolso,"41,936.000",4719,4719,sector económico en el mes de desembolso,"25,567.000"
7. Con 6 meses cumplidos,NaN,2908,2908,créditos recientes: aún no cumplen 6 meses,"1,811.000"


## 9. Comparación con la tabla de población de referencia

La tabla de referencia aplica el filtro de sector antes que los de producto. Por eso el factoring "saca 0" en ella: no tiene sector económico, así que el filtro de independientes ya lo había eliminado. Aquí los filtros de producto van primero, para que cada paso muestre lo que realmente elimina.

In [27]:
referencia = pd.DataFrame([
    ("Base cruda", 32082, final.loc["1. Base cruda", "clientes"]),
    ("Sin el corte reextraído", 30883, final.loc["2. Sin el corte reextraído", "clientes"]),
    ("Independientes + sin factoring + motos", 4719, final.loc["6. Independientes al desembolso", "clientes"]),
    ("Con 6 meses cumplidos", 2908, final.loc["7. Con 6 meses cumplidos", "clientes"]),
    ("Completan los 6 meses en el panel", 2529, int((obj["estado_6m"] == "observado").sum())),
], columns=["paso", "tabla de referencia", "este notebook"])
referencia["diferencia"] = referencia["este notebook"] - referencia["tabla de referencia"]
referencia

,paso,tabla de referencia,este notebook,diferencia
0,Base cruda,32082,32082,0
1,Sin el corte reextraído,30883,30883,0
2,Independientes + sin factoring + motos,4719,4719,0
3,Con 6 meses cumplidos,2908,2908,0
4,Completan los 6 meses en el panel,2529,2524,-5


In [28]:
# Composición de los créditos con resultado a 6 meses
obs = int((obj["estado_6m"] == "observado").sum())
sale = obj["estado_6m"].str.startswith("sale")
malos_salen = int(obj.loc[sale, "mora30_6m"].sum())
anticipados = int((sale & obj["mora30_6m"].isna()).sum())
print(f"Créditos que completan 6 meses en el panel:             {obs:,}")
print(f"+ Salen del panel con deterioro 30+ ya observado:        {malos_salen}")
print(f"+ Pagos anticipados (salen sin deterioro, buenos):       {anticipados}")
print(f"= Créditos con resultado a 6 meses:                      {obs + malos_salen + anticipados:,}")

Créditos que completan 6 meses en el panel:             2,524
+ Salen del panel con deterioro 30+ ya observado:        8
+ Pagos anticipados (salen sin deterioro, buenos):       376
= Créditos con resultado a 6 meses:                      2,908


**Diferencias con la tabla de referencia:**

- **Población de independientes con moto (4.719) y maduros a 6 meses (2.908):** coinciden exactamente. Los conteos intermedios difieren solo por el orden de los filtros.
- **Créditos que completan 6 meses: 2.524 aquí frente a 2.529 en la tabla.** La tabla arma el panel por **cliente**, así que incluye todas sus obligaciones. Aquí se arma por la **obligación de origen**. Hay 5 clientes cuyo primer crédito salió del panel antes de 6 meses y que tienen un segundo crédito que sí los superó. Con el panel por cliente, ese segundo crédito los marca como observados (celda siguiente). Filtrando por la obligación de origen, ambas construcciones coinciden exactamente.
- **Con resultado a 6 meses: 2.908 = 2.524 + 8 + 376**: los que completan el panel, los que salen **después** de alcanzar 30+ días de mora (eventos, porque su deterioro está observado) y los pagos anticipados (buenos pagadores).

In [29]:
# Los 5 clientes: la tabla de referencia arma el panel por cliente (todas sus obligaciones)
por_cliente = p1[p1["Cedula_Encriptado"].isin(p5["Cedula_Encriptado"])].copy()
por_cliente["ant"] = (fecha(por_cliente["Corte"]).dt.to_period("M") - fecha(por_cliente["Periodo_Des"]).dt.to_period("M")).apply(lambda x: x.n)
ant_cliente = por_cliente.groupby("Cedula_Encriptado")["ant"].max()
maduros_cli = set(p5.loc[p5["Numero_Encriptado"].isin(madura.index), "Cedula_Encriptado"])
observados_cli = set(p5.loc[p5["Numero_Encriptado"].isin(obj.index[obj["estado_6m"] == "observado"]), "Cedula_Encriptado"])
extra = sorted((set(ant_cliente.index[ant_cliente >= 6]) & maduros_cli) - observados_cli)
print(f"Observados con panel por cliente: {len(observados_cli) + len(extra):,} | por obligación de origen: {len(observados_cli):,} | diferencia: {len(extra)}")
detalle = por_cliente[por_cliente["Cedula_Encriptado"].isin(extra)].groupby(["Cedula_Encriptado", "Numero_Encriptado"]).agg(
    desembolso=("Periodo_Des", "min"), ultimo_corte=("Corte", "max"), meses_observados=("ant", "max")).reset_index()
detalle["credito"] = np.where(detalle["Numero_Encriptado"].isin(p5["Numero_Encriptado"]), "primero (el analizado)", "posterior")
detalle["cliente"] = detalle["Cedula_Encriptado"].rank(method="dense").astype(int)
detalle.assign(desembolso=fecha(detalle["desembolso"]).dt.strftime("%Y-%m"), ultimo_corte=fecha(detalle["ultimo_corte"]).dt.strftime("%Y-%m"))[
    ["cliente", "credito", "desembolso", "ultimo_corte", "meses_observados"]].sort_values(["cliente", "desembolso"])

Observados con panel por cliente: 2,529 | por obligación de origen: 2,524 | diferencia: 5


,cliente,credito,desembolso,ultimo_corte,meses_observados
1,1,primero (el analizado),2024-02,2024-06,4
0,1,posterior,2024-10,2025-12,14
2,2,primero (el analizado),2023-08,2023-08,0
3,2,posterior,2023-09,2024-11,14
4,3,primero (el analizado),2023-12,2024-02,2
5,3,posterior,2024-10,2025-07,9
7,4,primero (el analizado),2024-06,2024-09,3
6,4,posterior,2025-04,2025-12,8
9,5,primero (el analizado),2024-06,2024-09,3
8,5,posterior,2025-02,2025-12,10


In [30]:
# Verificación: el embudo coincide con el guardado en datos/embudo_poblacion.csv (no se reescribe)
guardado = pd.read_csv(RAIZ / "datos" / "embudo_poblacion.csv", encoding="utf-8-sig")
assert list(guardado["obligaciones"].astype(int)) == list(final["obligaciones"].astype(int))
print("OK: el embudo coincide con datos/embudo_poblacion.csv")

OK: el embudo coincide con datos/embudo_poblacion.csv
